In [0]:
DROP TABLE IF EXISTS silver.crm_cust_info;

CREATE TABLE silver.crm_cust_info
(
    cst_id INT,
    cst_key STRING,
    cst_firstname STRING,
    cst_lastname STRING,
    cst_marital_status STRING,
    cst_gndr STRING,
    cst_create_date DATE,
    ingestion_date TIMESTAMP
)
USING DELTA;


DROP TABLE IF EXISTS silver.crm_prd_info;

CREATE TABLE silver.crm_prd_info
(
    prd_id INT,
    cat_id STRING,
    prd_key STRING,
    prd_nm STRING,
    prd_cost DECIMAL(12,2),
    prd_line STRING,
    prd_start_dt DATE,
    prd_end_dt DATE,
    ingestion_date TIMESTAMP
)
USING DELTA;


DROP TABLE IF EXISTS silver.crm_sales_details;

CREATE TABLE silver.crm_sales_details
(
    sls_ord_num STRING,
    sls_prd_key STRING,
    sls_cust_id INT,
    sls_order_dt DATE,
    sls_ship_dt DATE,
    sls_due_dt DATE,
    sls_sales DECIMAL(12,2),
    sls_quantity INT,
    sls_price DECIMAL(12,2),
    ingestion_date TIMESTAMP
)
USING DELTA;




In [0]:
-- =========================================================================
-- Silver Layer: Bronze -> Silver 
-- Incremental and Cleansing (Deduplication, Normalization, Data Type Casting)
-- =========================================================================

In [0]:
--Load Silver.crm_cust_info 
-- STEP 1
UPDATE control.watermark
SET last_run_status='RUNNING',
    last_run_start=current_timestamp(),
    last_run_end=NULL
WHERE table_name='crm_cust_info' AND layer='silver';

-- STEP 2–3 (Watermark + Count)
WITH wm AS (
     SELECT COALESCE(last_load_timestamp, TIMESTAMP('1900-01-01')) AS last_load_timestamp
    FROM control.watermark
    WHERE table_name='crm_cust_info' AND layer='silver'
),
cnt AS (
    SELECT COUNT(*)
    FROM bronze.crm_cust_info
    WHERE cst_id IS NOT NULL
    and cst_id!='cst_id'
    AND ingestion_date > (SELECT max(last_load_timestamp) FROM wm)
)

MERGE INTO silver.crm_cust_info tgt
USING (
    WITH wm AS (
        SELECT max(last_load_timestamp) AS max_last_load_timestamp
        FROM control.watermark
        WHERE table_name='crm_cust_info' AND layer='silver'
    ),

    src AS (
        SELECT *
        FROM bronze.crm_cust_info
        WHERE cst_id IS NOT NULL
        and cst_id!='cst_id'
        AND ingestion_date > (SELECT max_last_load_timestamp FROM wm)
    ),

    dedup AS (
        SELECT
            CAST(cst_id AS INT) cst_id,
            cst_key,
            TRIM(cst_firstname) cst_firstname,
            TRIM(cst_lastname) cst_lastname,
            CASE WHEN UPPER(TRIM(cst_marital_status))='S' THEN 'Single'
                 WHEN UPPER(TRIM(cst_marital_status))='M' THEN 'Married'
                 ELSE 'n/a' END cst_marital_status,
            CASE WHEN UPPER(TRIM(cst_gndr))='F' THEN 'Female'
                 WHEN UPPER(TRIM(cst_gndr))='M' THEN 'Male'
                 ELSE 'n/a' END cst_gndr,
            CAST(cst_create_date AS DATE) cst_create_date,
            ingestion_date,
            ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC, ingestion_date DESC) rn
        FROM src
    )

    SELECT *,
        MD5(CONCAT_WS('|',
            COALESCE(CAST(cst_id AS STRING),''),
            COALESCE(cst_key,''),
            COALESCE(cst_firstname,''),
            COALESCE(cst_lastname,''),
            COALESCE(cst_marital_status,''),
            COALESCE(cst_gndr,''),
            COALESCE(CAST(cst_create_date AS STRING),'')
        )) AS customer_hash
    FROM dedup WHERE rn=1
) src
ON tgt.cst_id=src.cst_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

UPDATE control.watermark
SET last_load_timestamp=(SELECT MAX(ingestion_date) FROM bronze.crm_cust_info),
last_run_status='SUCCESS',
last_run_end=current_timestamp(),
updated_at=current_timestamp()
WHERE table_name='crm_cust_info' AND layer='silver';


In [0]:
--Load silver.crm_prd_info

MERGE INTO silver.crm_prd_info tgt
USING (
    WITH wm AS (
        SELECT COALESCE(last_load_timestamp, TIMESTAMP('1900-01-01')) AS last_load_timestamp
        FROM control.watermark
        WHERE table_name='crm_prd_info' AND layer='silver'
    ),

    src AS (
        SELECT *
        FROM bronze.crm_prd_info
        WHERE prd_id IS NOT NULL
        and prd_id!='prd_id'
        and ingestion_date > (SELECT max(last_load_timestamp) FROM wm)
    ),

    dedup AS (
        SELECT
             TRY_CAST(NULLIF(TRIM(prd_id), '') AS INT) AS prd_id,
            REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,
            REPLACE(SUBSTRING(prd_key,7,len(prd_key)), '-', '_') AS prd_key,
            TRIM(prd_nm) prd_nm,
            TRY_CAST(prd_cost AS DOUBLE) prd_cost,
            CASE WHEN UPPER(TRIM(prd_line))='M' THEN 'Mountain'
                 WHEN UPPER(TRIM(prd_line))='R' THEN 'Road'
                 WHEN UPPER(TRIM(prd_line))='T' THEN 'Touring'
                 ELSE 'n/a' END prd_line,
            CAST(prd_start_dt AS DATE) prd_start_dt,
            --CAST(prd_end_dt AS DATE) prd_end_dt,
             date_sub(
                        LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt), 1
                        ) AS prd_end_dt,
            ingestion_date,
            ROW_NUMBER() OVER (PARTITION BY prd_key, prd_start_dt ORDER BY ingestion_date DESC) rn
        FROM src
    )

    SELECT *,
        MD5(CONCAT_WS('|',
            COALESCE(prd_key,''),
            COALESCE(prd_nm,''),
            COALESCE(CAST(prd_cost AS STRING),''),
            COALESCE(prd_line,''),
            COALESCE(CAST(prd_start_dt AS STRING),''),
            COALESCE(CAST(prd_end_dt AS STRING),'')
        )) AS product_hash
    FROM dedup WHERE rn=1
) src
ON tgt.prd_key=src.prd_key AND tgt.prd_start_dt=src.prd_start_dt
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

UPDATE control.watermark
SET last_load_timestamp=(SELECT MAX(ingestion_date) FROM bronze.crm_prd_info),
last_run_status='SUCCESS',
last_run_end=current_timestamp(),
updated_at=current_timestamp()
WHERE table_name='crm_prd_info' AND layer='silver';

In [0]:
--sales details version 2
-- STEP 1: SET RUNNING
UPDATE control.watermark
SET last_run_status = 'RUNNING',
    last_run_start = current_timestamp(),
    last_run_end = NULL
WHERE table_name = 'crm_sales_details'
AND layer = 'silver';

-- STEP 2: READ WATERMARK SAFELY
WITH wm AS (
    SELECT COALESCE(last_load_timestamp, TIMESTAMP('1900-01-01')) AS last_load_timestamp
    FROM control.watermark
    WHERE table_name = 'crm_sales_details'
    AND layer = 'silver'
),

-- STEP 3: SOURCE DATA
src AS (
    SELECT *
    FROM bronze.crm_sales_details
    WHERE sls_ord_num IS NOT NULL
    AND sls_ord_num != 'sls_ord_num'
    AND ingestion_date > (SELECT last_load_timestamp FROM wm)
),

-- STEP 4: COUNT BEFORE LOAD ✅
cnt AS (
    SELECT COUNT(*) AS record_count FROM src
),

-- STEP 5: TRANSFORM
casted AS (
    SELECT
        sls_ord_num,
        REPLACE(sls_prd_key,'-','_') AS sls_prd_key,
        CAST(sls_cust_id AS INT) AS sls_cust_id,
        TRY_TO_DATE(sls_order_dt,'yyyyMMdd') AS sls_order_dt,
        TRY_TO_DATE(sls_ship_dt,'yyyyMMdd') AS sls_ship_dt,
        TRY_TO_DATE(sls_due_dt,'yyyyMMdd') AS sls_due_dt,
        CAST(sls_quantity AS INT) AS sls_quantity,
        CAST(sls_price AS DOUBLE) AS raw_price,
        CAST(sls_sales AS DOUBLE) AS raw_sales,
        ingestion_date
    FROM src
),

final AS (
    SELECT *,
        -- Corrected sales
        CASE 
            WHEN raw_sales IS NULL OR raw_sales <= 0 
                 OR raw_sales != (sls_quantity * ABS(raw_price))
            THEN sls_quantity * ABS(raw_price)
            ELSE raw_sales
        END AS sls_sales,

        -- Corrected price (IMPORTANT FIX)
        CASE 
            WHEN raw_price IS NULL OR raw_price <= 0 THEN
                CASE 
                    WHEN sls_quantity = 0 THEN NULL
                    ELSE (sls_quantity * ABS(raw_price)) / sls_quantity
                END
            ELSE raw_price
        END AS sls_price
    FROM casted
),

dedup AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY sls_ord_num, sls_prd_key, sls_cust_id
            ORDER BY ingestion_date DESC
        ) rn
    FROM final
),

-- FINAL SOURCE
final_src AS (
    SELECT
        sls_ord_num, sls_prd_key, sls_cust_id,
        sls_order_dt, sls_ship_dt, sls_due_dt,
        sls_sales, sls_quantity, sls_price,
        ingestion_date,

        MD5(CONCAT_WS('|',
            COALESCE(sls_ord_num,''), 
            COALESCE(sls_prd_key,''),
            COALESCE(CAST(sls_cust_id AS STRING),''),
            COALESCE(CAST(sls_sales AS STRING),''),
            COALESCE(CAST(sls_quantity AS STRING),''),
            COALESCE(CAST(sls_price AS STRING),'')
        )) AS sales_hash
    FROM dedup
    WHERE rn = 1
)

-- STEP 6: MERGE
MERGE INTO silver.crm_sales_details tgt
USING final_src src
ON tgt.sls_ord_num = src.sls_ord_num
AND tgt.sls_prd_key = src.sls_prd_key
AND tgt.sls_cust_id = src.sls_cust_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- STEP 7: UPDATE WATERMARK (CORRECT)
WITH wm AS (
    SELECT COALESCE(last_load_timestamp, TIMESTAMP('1900-01-01')) AS last_load_timestamp
    FROM control.watermark
    WHERE table_name = 'crm_sales_details'
    AND layer = 'silver'
),

cnt AS (
    SELECT COUNT(*) AS record_count
    FROM bronze.crm_sales_details
    WHERE sls_ord_num IS NOT NULL
    AND sls_ord_num != 'sls_ord_num'
    AND ingestion_date > (SELECT last_load_timestamp FROM wm)
),

mx AS (
    SELECT MAX(ingestion_date) AS max_dt
    FROM bronze.crm_sales_details
)

UPDATE control.watermark
SET 
    last_load_timestamp = (SELECT max_dt FROM mx),
    records_processed = (SELECT record_count FROM cnt),
    last_run_status = 'SUCCESS',
    last_run_end = current_timestamp(),
    updated_at = current_timestamp()
WHERE table_name = 'crm_sales_details'
AND layer = 'silver';